# Parameter Golf: Gimlet Hetero vs. Uniform (Non-Record Eval)
This notebook verifies feasibility for non-record submission by performing an end-to-end training, export (with explicit zstd compression over the default zlib), and BPB evaluation for both the `gimlet-hetero` layout and a unified baseline layout.

In [ ]:
%%bash
# 1. Setup Environment, Data, and Repositories
rm -rf openai-pg pg
git clone https://github.com/openai/parameter-golf.git openai-pg
git clone https://github.com/jmoncayo-pursuit/parameter-golf-gimlet-hetero.git pg

cd pg
pip install -q -r requirements.txt zstandard sentencepiece

# Prepare single shard for Colab
python ../openai-pg/data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

## 2. Force zstd Compression & Prepare Baselines
The core export pipeline in `train_gpt.py` originally uses `zlib`. We update it to leverage `zstd` with maximum compression (`level=22`). We also create a uniform-layer analog to measure the impact of the heterogeneous architecture.

In [ ]:
import re

with open('pg/train_gpt.py', 'r') as f:
    code = f.read()

# 1. Force zstd usage for zlib compression
code = "import zstandard as zstd\n" + code
code = re.sub(
    r'zlib\.compress\(quant_raw,\s*level=\d+\)',
    'zstd.ZstdCompressor(level=22).compress(quant_raw)',
    code
)
code = re.sub(
    r'zlib\.decompress\(quant_blob_disk\)',
    'zstd.ZstdDecompressor().decompress(quant_blob_disk)',
    code
)

with open('pg/train_gpt_hetero.py', 'w') as f:
    f.write(code)

# 2. Create the Uniform Baseline version
old_func = """def get_layer_config(i: int, total_layers: int) -> tuple[float, int]:
    """Returns (mlp_mult, export_clip_val) for a given layer index."""
    if i < EARLY_LAYER_COUNT:
        return 3.0, 31
    elif i >= total_layers - LATE_LAYER_COUNT:
        return 3.0, 31
    else:
        return 4.5, 7"""

new_func = """def get_layer_config(i: int, total_layers: int) -> tuple[float, int]:
    """Uniform baseline corresponding to equivalent FLOPs."""
    return 3.681818, 127"""

if old_func in code:
    uniform_code = code.replace(old_func, new_func)
else:
    print("Warning: Could not patch get_layer_config exactly as expected.")
    uniform_code = code

with open('pg/train_gpt_uniform.py', 'w') as f:
    f.write(uniform_code)

print("Train scripts generated: pg/train_gpt_hetero.py, pg/train_gpt_uniform.py")


## 3. Train Gimlet Hetero
Runs 500 iterations. We disable `torch.compile` (`TORCHDYNAMO_DISABLE=1`) to prevent hangs/OOM on Colab environments.

In [ ]:
%%bash
cd pg
export TORCHDYNAMO_DISABLE=1
export DATA_PATH=../openai-pg/data/datasets/fineweb10B_sp1024
export TOKENIZER_PATH=../openai-pg/data/tokenizers/fineweb_1024_bpe.model
export VOCAB_SIZE=1024
export ITERATIONS=500
export WARMUP_STEPS=50
export TRAIN_BATCH_TOKENS=16384

echo "Starting Hetero Training..."
python train_gpt_hetero.py > hetero_log.txt 2>&1
echo "Hetero Training Complete."


## 4. Train Uniform Baseline


In [ ]:
%%bash
cd pg
export TORCHDYNAMO_DISABLE=1
export DATA_PATH=../openai-pg/data/datasets/fineweb10B_sp1024
export TOKENIZER_PATH=../openai-pg/data/tokenizers/fineweb_1024_bpe.model
export VOCAB_SIZE=1024
export ITERATIONS=500
export WARMUP_STEPS=50
export TRAIN_BATCH_TOKENS=16384

echo "Starting Uniform Training..."
python train_gpt_uniform.py > uniform_log.txt 2>&1
echo "Uniform Training Complete."


## 5. Summary Report


In [ ]:
import re
from pathlib import Path

def parse_logs(logfile):
    if not Path(logfile).exists():
        return 0, 0.0, 0.0, "File missing"
    
    with open(logfile, "r") as f:
        log = f.read()
    
    status = "Stable"
    if 'train_loss:nan' in log or 'train_loss:inf' in log:
        status = "Diverged (NaN/Inf)"
    elif "final_int8_zlib_roundtrip_exact" not in log:
        status = "Failed to complete"
        
    size_match = re.search(r'Total submission size int8\+zlib:\s+(\d+)\s+bytes', log)
    size = int(size_match.group(1)) if size_match else 0
    
    bpb_match = re.search(r'final_int8_zlib_roundtrip_exact.*?val_bpb:([0-9.]+)', log)
    bpb = float(bpb_match.group(1)) if bpb_match else 0.0
    
    ratio_match = re.search(r'payload_ratio:([0-9.]+)x', log)
    ratio = float(ratio_match.group(1)) if ratio_match else 0.0
    
    return size, bpb, ratio, status

LIMIT = 16_000_000
h_size, h_bpb, h_ratio, h_status = parse_logs("pg/hetero_log.txt")
u_size, u_bpb, u_ratio, u_status = parse_logs("pg/uniform_log.txt")

h_headroom = LIMIT - h_size
u_headroom = LIMIT - u_size

print(f"{'Metric':<25} | {'Heterogeneous (Gimlet)':<25} | {'Uniform Baseline':<20}")
print("-" * 75)
print(f"{'Artifact Size (MB)':<25} | {h_size / 1e6:<25.3f} | {u_size / 1e6:<20.3f}")
print(f"{'Headroom (<16MB)':<25} | {h_headroom / 1e6:<25.3f} | {u_headroom / 1e6:<20.3f}")
print(f"{'Sliding BPB':<25} | {h_bpb:<25.4f} | {u_bpb:<20.4f}")
print(f"{'Compression Ratio':<25} | {h_ratio:<25.2f} | {u_ratio:<20.2f}")
print(f"{'Stability':<25} | {h_status:<25} | {u_status:<20}")
print("-" * 75)

if h_size > 0:
    if h_size <= LIMIT:
        print(f"\n✅ Hetero model qualifies for non-record submission! It is {h_headroom/1e6:.2f}MB under the limit.")
    else:
        print(f"\n❌ Hetero model exceeds 16MB by {-h_headroom/1e6:.2f}MB.")
